<a href="https://colab.research.google.com/github/BraedynL0530/Adveretisement-automation/blob/main/PaperProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from transformers import OwlViTForObjectDetection, OwlViTImageProcessor, OwlViTProcessor
from PIL import Image

# Use the full processor (not just image_processor)
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch16")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch16")

# Enable hidden states for object queries
model.config.output_hidden_states = True

image = Image.open("/content/download.jpg")
text_queries = ["dog","human hand", "grass"]  # or ["person", "cup", "hand", ...]

# Process image and text together
inputs = processor(images=image, text=text_queries, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Bounding boxes - normalized cxcywh format [0,1]
boxes = outputs.pred_boxes  # shape: (1, N_queries, 4)

# Object query embeddings from the last decoder layer
object_queries = outputs.vision_model_output.hidden_states[-1]   # shape: (1, N_queries, hidden_dim)

# Confidence scores (logits)
logits = outputs.logits  # shape: (1, N_queries, len(text_queries))

# Filter by confidence threshold
conf_threshold = 0.1
scores = torch.sigmoid(logits).max(dim=-1).values  # (1, N_queries)
valid = scores > conf_threshold

print(f"Found {valid.sum().item()} objects")
print(f"Boxes shape: {boxes.shape}")
print(f"Queries shape: {object_queries.shape}")

Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch16
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
owlvit.vision_model.embeddings.position_ids | UNEXPECTED |  | 
owlvit.text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 3 objects
Boxes shape: torch.Size([1, 2304, 4])
Queries shape: torch.Size([1, 2305, 768])


In [ ]:
class linearProjector(nn.Module): #turns bounding box + class into readable inputs for nlp
  def __init__(self,query_dim, bbox_dim, output_dim):
    super().__init__()
    self.bbox_embed = nn.Sequential(
        nn.Linear(bbox_dim,64),
        ReLu(),
        nn.Linear(64,query_dim)
    )
    self.proj = nn.Linear(query_dim, output_dim)

  def forward(self,object_queries,bboxes):
    spatial_emb = self.bbox_embed(bboxes)
    enriched = spatial_emb+object_queries
    tokens = self.proj(enriched)
    return tokens


In [ ]:
class worldPhsyicsModel(nn.Module): #TBD summrizer -> Vllm core needs to have normlilzed outputs first.
  def __init__(self,):
    super().__init__()


  def forward(self,)